In [5]:
import torch
# 接精准引入 GPT2 的专属分词类，不再用 AutoTokenizer 乱猜
from transformers import GPT2Tokenizer, AutoModelForCausalLM

print("正在检查 Windows 电脑设备...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前使用的计算设备是: {device.upper()}")

print("\n正在从本地 my_model 文件夹离线加载工具人模型...")
model_path = r"D:\Users\qq858\Desktop\TDQE_Project\my_model"

try:
    # 🎯 用专属的 GPT2Tokenizer 离线加载，直接断了它找 sentencepiece 的念头
    tokenizer = GPT2Tokenizer.from_pretrained(
        model_path, 
        local_files_only=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    
    # 离线加载模型核心
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        local_files_only=True
    ).to(device)
    
    print("\n✨ 恭喜！模型已从本地成功加载，准备工作全部就绪！")
except Exception as e:
    print(f"\n❌ 加载失败，请确认你的文件夹名字和文件放对了吗？\n错误信息：{e}")

正在检查 Windows 电脑设备...
当前使用的计算设备是: CPU

正在从本地 my_model 文件夹离线加载工具人模型...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


✨ 恭喜！模型已从本地成功加载，准备工作全部就绪！


正在运行论文核心公式(1)计算中...

--- 测试样本 A（高质量论文文本） ---
📈 样本 A 鲁棒性一致性得分: 0.0002

--- 测试样本 B（低质量乱码文本） ---
📉 样本 B 鲁棒性一致性得分: 0.0000



In [4]:
!pip install sentencepiece tiktoken transformers --upgrade

正在运行双重指标检测中...

--- 📊 测试样本 A（高质量论文文本） ---


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


🛡️ 鲁棒性得分: 0.0061
🎯 准确性置信度: 0.8309

--- 📊 测试样本 B（低质量乱码文本） ---
🛡️ 鲁棒性得分: 0.0026
🎯 准确性置信度: 0.0708



--- 🔬 完完全全对齐论文描述的指标计算 ---
[样本 A - 高质量文本]
🎯 准确性置信度 (长度加权): 0.0331
🛡️ 鲁棒性得分 (公式1还原): 0.1175

[样本 B - 低质量乱码]
🎯 准确性置信度 (长度加权): 0.0005
🛡️ 鲁棒性得分 (公式1还原): 0.2133


In [ ]:
import numpy as np
import torch
import pandas as pd
import os
import re
import time

# ========================================================
# 1. 📂 扫描 D:\Downloads\archive 提取并深度净化全量文章
# ========================================================
folder_path = r"D:\Downloads\archive"

print("📂 正在轰炸式扫描 20 个大 TXT 文件，提取完整 20NG 全量文章...")
all_real_news = []

# 💡 核心修复点：定义一个清理函数，彻底过滤掉会导致 Excel 报错的不可见非法控制字符
def clean_illegal_chars(text):
    if not isinstance(text, str):
        return ""
    # 过滤掉 ASCII 码在 0-31 之间的控制字符（保留换行符 \n 和制表符 \t）
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)

for file_name in os.listdir(folder_path):
    if file_name.endswith('.txt') and file_name != 'requirements.txt':
        file_key = file_name.replace('.txt', '')
        full_file_path = os.path.join(folder_path, file_name)
        
        try:
            with open(full_file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                
            articles = re.split(r'\n(?=From:|Subject:|Newsgroups:|Path:)', content)
            
            for art in articles:
                clean_art = art.strip()
                if len(clean_art) > 150:
                    # 💡【深度净化】：在存入矩阵前，全自动剔除所有古代低位控制字符
                    clean_art = clean_illegal_chars(clean_art)
                    all_real_news.append({
                        'category': file_key,
                        'text': clean_art
                    })
        except Exception as e:
            pass

df = pd.DataFrame(all_real_news)
print(f"\n✅ 原始数据净化完毕！总共成功提取出 【{len(df)}】 篇无噪声的全量官方文章！")
print("🚀 完完整整的 TDQE 全量质检长跑大引擎，现在正式启动...")

# ========================================================
# 2. 🔬 完完全全对齐复旦大学论文的核心指标计算函数 (TDQE框架)
# ========================================================
def calculate_paper_metrics(text, m=3):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    if inputs['input_ids'].size(1) < 5:
        return 0.0, 0.0
        
    model.train() 
    embeddings = []
    with torch.no_grad():
        for _ in range(m):
            outputs = model(**inputs, output_hidden_states=True)
            v_i = outputs.hidden_states[-1].mean(dim=1).cpu().numpy()[0]
            embeddings.append(v_i)
            
    distances = []
    for i in range(m):
        for j in range(i + 1, m):
            distances.append(np.linalg.norm(embeddings[i] - embeddings[j]))
    avg_distance = np.mean(distances)
    
    robustness_score = 1 / (1 + np.exp(avg_distance)) * 0.5
    
    model.eval() 
    with torch.no_grad():
        summary_ids = model.generate(
            inputs['input_ids'], 
            max_new_tokens=80, 
            num_beams=2, 
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id
        )
        summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    tokens_t = set(text.lower().split())
    tokens_s = set(summary_text.lower().split())
    
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'is', 'are', 'was', 'were', 'to', 'of', 'in', 'on', 'for', 'with'}
    tokens_t = tokens_t - stop_words
    tokens_s = tokens_s - stop_words
    
    accuracy_score = 0.5 * (len(tokens_t & tokens_s) / len(tokens_t | tokens_s)) if tokens_s else 0.0
        
    return robustness_score, accuracy_score

# ========================================================
# 3. 🏃‍♂️ 开启全量长跑（每 50 条播报一次进度）
# ========================================================
if len(df) > 0:
    robustness_list = []
    accuracy_list = []
    final_quality_list = []
    
    start_time = time.time()

    for idx, row in df.iterrows():
        r_score, a_score = calculate_paper_metrics(str(row['text']))
        q_score = r_score + a_score
        
        robustness_list.append(r_score)
        accuracy_list.append(a_score)
        final_quality_list.append(q_score)
        
        if (idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / (idx + 1)
            remaining_hours = (avg_time * (len(df) - (idx + 1))) / 3600
            print(f"⏳ 全量大长跑当前进度: {idx + 1}/{len(df)} 篇... 预计还需要挂机: {remaining_hours:.1f} 小时")
            
            # 💡 此时导出的临时 Excel 再也不会被非法字符阻挡了
            if (idx + 1) % 200 == 0:
                temp_df = df.head(idx + 1).copy()
                temp_df['🛡️ 鲁棒性得分'] = robustness_list
                temp_df['🎯 准确性得分'] = accuracy_list
                temp_df['✨ TDQE总质量分数'] = final_quality_list
                temp_df.to_excel(os.path.join(folder_path, "TDQE_全量实验_临时安全备份.xlsx"), index=False)

    df['所属新闻主题'] = df['category']
    df['新闻正文预览'] = df['text'].str.slice(0, 150) + "..."
    df['🛡️ 鲁棒性得分'] = robustness_list
    df['🎯 准确性得分'] = accuracy_list
    df['✨ TDQE总质量分数'] = final_quality_list

    df_sorted = df[['所属新闻主题', '新闻正文预览', '🛡️ 鲁棒性得分', '🎯 准确性得分', '✨ TDQE总质量分数']].sort_values(by='✨ TDQE总质量分数', ascending=False)

    # ========================================================
    # 4. 💾 导出终极实验报告
    # ========================================================
    output_file = f"TDQE_20NG官方全量_{len(df)}条_终极实验报告.xlsx"
    df_sorted.to_excel(os.path.join(folder_path, output_file), index=False)

    print(f"\n🎉 【史诗级神迹】一万八千条全量真实长篇新闻大长跑全部圆满胜利结束！！")
    print(f"💾 最终成果大报告已保存至：【 {os.path.join(folder_path, output_file)} 】")

📂 正在轰炸式扫描 20 个大 TXT 文件，提取完整 20NG 全量文章...

✅ 原始数据净化完毕！总共成功提取出 【39398】 篇无噪声的全量官方文章！
🚀 完完整整的 TDQE 全量质检长跑大引擎，现在正式启动...
⏳ 全量大长跑当前进度: 50/39398 篇... 预计还需要挂机: 34.2 小时
⏳ 全量大长跑当前进度: 100/39398 篇... 预计还需要挂机: 35.1 小时
⏳ 全量大长跑当前进度: 150/39398 篇... 预计还需要挂机: 35.3 小时
⏳ 全量大长跑当前进度: 200/39398 篇... 预计还需要挂机: 35.6 小时
⏳ 全量大长跑当前进度: 250/39398 篇... 预计还需要挂机: 35.1 小时
⏳ 全量大长跑当前进度: 300/39398 篇... 预计还需要挂机: 35.2 小时
⏳ 全量大长跑当前进度: 350/39398 篇... 预计还需要挂机: 35.4 小时
⏳ 全量大长跑当前进度: 400/39398 篇... 预计还需要挂机: 35.5 小时
⏳ 全量大长跑当前进度: 450/39398 篇... 预计还需要挂机: 35.3 小时
⏳ 全量大长跑当前进度: 500/39398 篇... 预计还需要挂机: 35.3 小时
⏳ 全量大长跑当前进度: 550/39398 篇... 预计还需要挂机: 34.8 小时
⏳ 全量大长跑当前进度: 600/39398 篇... 预计还需要挂机: 34.8 小时
⏳ 全量大长跑当前进度: 650/39398 篇... 预计还需要挂机: 34.7 小时
⏳ 全量大长跑当前进度: 700/39398 篇... 预计还需要挂机: 34.6 小时
⏳ 全量大长跑当前进度: 750/39398 篇... 预计还需要挂机: 34.7 小时
⏳ 全量大长跑当前进度: 800/39398 篇... 预计还需要挂机: 34.6 小时
⏳ 全量大长跑当前进度: 850/39398 篇... 预计还需要挂机: 34.6 小时
⏳ 全量大长跑当前进度: 900/39398 篇... 预计还需要挂机: 34.6 小时
⏳ 全量大长跑当前进度: 950/39398 篇... 预计还需要挂机: 34.6 小时
⏳ 全量大长跑当前进度: 1000/39398 篇... 预计

In [13]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpy